# DRAFT: Qwen3.5 NVARC private inference
Requires a new exact owner authorization and new ARC-SFT/runtime Datasets. This is a static development artifact; do not push under an earlier trial decision.


In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false', WANDB_DISABLED='true', PYTHONUNBUFFERED='1')
def exact(label, candidates):
    matches = [path for path in candidates if path.exists()]
    if len(matches) != 1: raise RuntimeError(f'{label} must resolve exactly once: {matches}')
    return matches[0]
base = exact('base Dataset', [Path('/kaggle/input/datasets/danielwuu/irisx-arc-agi-2-trial-002-smoke-001'), Path('/kaggle/input/irisx-arc-agi-2-trial-002-smoke-001')])
runtime = exact('new runtime Dataset', [Path('/kaggle/input/datasets/DRAFT-OWNER/new-nvarc-runtime-bundle'), Path('/kaggle/input/new-nvarc-runtime-bundle')])
model = exact('new ARC-SFT Dataset', [Path('/kaggle/input/datasets/DRAFT-OWNER/new-nvarc-arc-sft-model'), Path('/kaggle/input/new-nvarc-arc-sft-model')])
competition = exact('official competition challenge', [Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json'), Path('/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json')])
manifest = json.loads((runtime / 'staging-manifest.json').read_text())
def check(path, digest):
    if hashlib.sha256(path.read_bytes()).hexdigest() != digest.removeprefix('sha256:'): raise RuntimeError(f'bound digest differs: {path.name}')
outbound = runtime / 'outbound.zip'
wheel = runtime / manifest['wheel_name']
check(outbound, manifest['outbound_digest'])
check(wheel, manifest['wheel_digest'])
authorization = runtime / 'authorization.json'
if not authorization.is_file(): raise RuntimeError('new owner authorization file is absent')
auth = json.loads(authorization.read_text())
if auth.get('approved') is not True or not auth.get('owner_decision'): raise RuntimeError('draft authorization cannot run')
deps = []
for item in manifest['dependency_wheels']:
    path = runtime / item['name']; check(path, item['digest']); deps.append(str(path))
if not any(Path(path).name.startswith('peft-') for path in deps): raise RuntimeError('pinned offline PEFT wheel is required for TTT')
transformers_wheel = base / 'transformers-5.8.1-py3-none-any.whl'
if not transformers_wheel.is_file(): raise RuntimeError('pinned offline Transformers wheel is absent')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--force-reinstall', str(transformers_wheel), *deps, str(wheel)], check=True)
command = [sys.executable, '-u', '-m', 'irisx.arc_nvarc_kaggle', '--outbound', str(outbound), '--model-root', str(model), '--competition-source', str(competition), '--output', '/kaggle/working/result.zip', '--checkpoint', '/kaggle/working/checkpoint.json', '--authorization', str(authorization)]
subprocess.run(command, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
